# Phase 5: Baseline Model Evaluation

In this notebook, we present the evaluation of baseline machine learning models for the Business Entity Resolution task.
We have established the following evaluation setup in `run_phase5.py`:
- Leakage-safe train/validation split by `source1_entity_id`.
- Feature extraction using RapidFuzz for Name and Address comparisons.
- Evaluation metric: Macro F0.5, with close attention to precision and singleton accuracy.

The models evaluated were:
1. **Logistic Regression (Plain)**
2. **Logistic Regression (Balanced)**
3. **Random Forest Classifier**

## 1. Experimental Results

We tested the models on a validation set comprising 100 S1 entities (with 57.0% singletons). The candidate pairs for these entities were generated using an unconstrained blocking pipeline to simulate a high-recall candidate generation pool. 

| Model                     | Macro F0.5 | Macro Precision | Macro Recall | Singleton Accuracy | Positive-Match Precision | Runtime |
|---------------------------|------------|-----------------|--------------|--------------------|--------------------------|---------|
| **LogReg (Plain)**        | **0.9451** | **0.9600**      | 0.8900       | **1.0000**         | 0.9070                   | 0.14s   |
| **LogReg (Balanced)**     | 0.2801     | 0.2520          | 0.5055       | 0.1579             | 0.3768                   | 0.15s   |
| **RandomForest**          | 0.9267     | 0.9322          | **0.9052**   | 0.9474             | **0.9121**               | 4.64s   |

## 2. Feature Ablation

To understand feature importance, we ran an ablation study using the LogReg (Balanced) setup:
- **Name Features Only**: F0.5 = 0.2360
- **Address & Country Features Only**: F0.5 = 0.2868

While address strings provide a marginally better signal on their own, the full feature set (F0.5 = 0.2801 on Balanced) indicates that both Name and Address contribute distinct signals for predicting matches.

## 3. Conclusion & Selection for Phase 6

**Selected Model: Logistic Regression (Plain)**

**Reasoning:**
The LogReg (Plain) model significantly outperforms the other approaches on **Macro F0.5 (0.9451)**. Because the dataset is highly imbalanced (most candidate pairs are negative, and over half the S1 entities are singletons), preserving precision is critical. 

The LogReg (Plain) model achieves **100% Singleton Accuracy** on the validation set. This means it aggressively filters out false positives and correctly refuses to match singletons. While RandomForest achieved a marginally higher Macro Recall (0.9052 vs 0.8900), it sacrificed precision and singleton accuracy (0.9474), resulting in a lower F0.5 overall (0.9267).

The LogReg (Balanced) approach is overly eager to predict matches (Singleton Accuracy of just 0.1579), completely destroying precision and proving unsuitable for an F0.5-optimized task.

Therefore, LogReg (Plain) establishes a very strong and robust baseline.

**Deferred to Phase 6:**
In Phase 6, we will implement LightGBM, hard negative mining to expose the model to near-miss negatives, and a threshold search specifically tuned to optimize F0.5 (rather than using the default 0.5 probability cutoff) and improve singleton identification.